# UAD-Mobility: Uncertainty-Aware Dynamic Mobility Forecasting and Disability–Recovery Modelling
**Longitudinal bilateral knee osteoarthritis analysis (OAI progression cohort)**

This notebook is an executable research implementation using participant-level longitudinal records. It does **not** include synthetic/pseudo-patient data. Configure the input files and column mapping to match the OAI release you are authorised to use.

**Primary analysis:** V03 landmark → V06 target; bilateral WOMAC function as primary outcomes; secondary mobility outcomes and event-risk heads.  
**Important:** OAI files are not bundled. Obtain them through the authorised OAI data-access route. The notebook deliberately stops if required real fields are absent. Some equations in the supplied methods text lost their symbols/parameter definitions during formatting; where exact numerical specifications were absent, the implementation uses documented operational choices (not claims of exact equation reproduction).

## 0. Environment and reproducibility

In [ ]:
# Colab: run this cell once.
!pip -q install pandas numpy scipy scikit-learn matplotlib seaborn openpyxl lifelines

In [ ]:
import os, re, json, math, random, warnings, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, f1_score, roc_auc_score,
                             average_precision_score, brier_score_loss,
                             classification_report, confusion_matrix)
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Data locations and canonical schema

The pipeline expects either one already-merged long-format CSV/Excel file, or multiple files that you merge before this step. One row represents one participant × visit × knee. Participant-level measurements are repeated across the two knee rows (or can be merged into both rows).

Required canonical fields:

- `participant_id`, `visit`, `month`, `knee` (`R`/`L`)
- `womac_function` (0–68), `womac_pain`, `womac_stiffness`, `kl_grade`, `jsw`
- participant-level: `walk_speed`, `chair_completed` (0/1), `chair_pace`
- covariates: `bmi`, `comorbidity`, `physical_activity`, `age`, `sex`, `medication`
- optional dated events: `replacement_date`, `death_date`

If a variable is unavailable, do not fabricate it. Remove it from the feature list and update the required-field checks. Medication inventory absence is **unknown**, not non-use.

In [ ]:
# EDIT THESE PATHS. Example: mount Google Drive in Colab if required.
# from google.colab import drive
# drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/OAI/oai_longitudinal.csv"  # .csv, .xlsx, or .parquet
OUTPUT_DIR = "/content/uad_mobility_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Map your source column names to canonical names. Identity mappings are shown.
COLUMN_MAP = {
    "participant_id":"participant_id", "visit":"visit", "month":"month", "knee":"knee",
    "womac_function":"womac_function", "womac_pain":"womac_pain",
    "womac_stiffness":"womac_stiffness", "kl_grade":"kl_grade", "jsw":"jsw",
    "walk_speed":"walk_speed", "chair_completed":"chair_completed", "chair_pace":"chair_pace",
    "bmi":"bmi", "comorbidity":"comorbidity", "physical_activity":"physical_activity",
    "age":"age", "sex":"sex", "medication":"medication",
    "replacement_date":"replacement_date", "death_date":"death_date"
}

## 2. Load and harmonise actual data

In [ ]:
def read_table(path):
    p = str(path)
    if p.lower().endswith(".csv"): return pd.read_csv(p)
    if p.lower().endswith((".xlsx",".xls")): return pd.read_excel(p)
    if p.lower().endswith(".parquet"): return pd.read_parquet(p)
    raise ValueError("Use CSV, Excel, or Parquet input.")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Real OAI input not found: {DATA_PATH}. Upload/place your authorised OAI export and update DATA_PATH."
    )
raw = read_table(DATA_PATH)
rename_map = {src: dst for src, dst in COLUMN_MAP.items() if src in raw.columns}
df = raw.rename(columns=rename_map).copy()

required = ["participant_id","visit","month","knee","womac_function",
            "walk_speed","chair_completed","chair_pace"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required canonical columns: {missing}. Update COLUMN_MAP or provide fields.")
df["knee"] = df["knee"].astype(str).str.upper().replace(
    {"RIGHT":"R","LEFT":"L","1":"R","2":"L","0":"R"}
)
df = df[df["knee"].isin(["R","L"])].copy()
df["month"] = pd.to_numeric(df["month"], errors="coerce")
for c in ["womac_function","womac_pain","womac_stiffness","kl_grade","jsw",
          "walk_speed","chair_completed","chair_pace","bmi","comorbidity",
          "physical_activity","age"]:
    if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
for c in ["replacement_date","death_date"]:
    if c in df: df[c] = pd.to_datetime(df[c], errors="coerce")
df["participant_id"] = df["participant_id"].astype(str)
df["visit"] = df["visit"].astype(str).str.upper()
print("Rows:", len(df), "Participants:", df.participant_id.nunique())
display(df.head())

## 3. Participant-level consistency and observed functional registry (F0/F1/F2)

In [ ]:
# Participant-level fields may be repeated over knees. Collapse safely by participant/visit.
person_cols = [c for c in ["walk_speed","chair_completed","chair_pace","bmi",
                           "comorbidity","physical_activity","age","sex","medication",
                           "replacement_date","death_date"] if c in df.columns]
person = (df.groupby(["participant_id","visit","month"], as_index=False)[person_cols]
          .first()) if person_cols else df[["participant_id","visit","month"]].drop_duplicates()

# Join participant measures back to knee records.
knee_cols = [c for c in ["womac_function","womac_pain","womac_stiffness","kl_grade","jsw"] if c in df]
knee = df[["participant_id","visit","month","knee"]+knee_cols].drop_duplicates(
    ["participant_id","visit","month","knee"])
long = knee.merge(person, on=["participant_id","visit","month"], how="left")

# Operational registry. F0: both knee WOMAC <= 21.08, walk >= 1.26 m/s,
# chair pace >= 1/14 stands/s. F2: severe threshold (WOMAC >= 42.5, walk <1.13,
# chair pace <1/14) or documented physical inability. F1 otherwise, only if observed.
# These are study operational anchors, not a validated clinical classification.
wide = long.pivot_table(index=["participant_id","visit","month"],
                        columns="knee", values="womac_function", aggfunc="first").reset_index()
wide.columns = [f"womac_{c}" if c in ["R","L"] else c for c in wide.columns]
reg = person.merge(wide, on=["participant_id","visit","month"], how="left")
for c in ["womac_R","womac_L"]:
    if c not in reg: reg[c] = np.nan
registry_required = reg[["womac_R","womac_L","walk_speed","chair_completed"]].notna().all(axis=1)
# pace is required only for completed chair tests; failed tests count as severe evidence if documented.
chair_obs = reg["chair_completed"].notna()
chair_pace_ok = ((reg["chair_completed"] == 0) |
                 ((reg["chair_completed"] == 1) & reg["chair_pace"].notna()))
registry_required &= chair_obs & chair_pace_ok

severe = (
    (reg[["womac_R","womac_L"]].max(axis=1) >= 42.5) |
    (reg["walk_speed"] < 1.13) |
    ((reg["chair_completed"] == 1) & (reg["chair_pace"] < 1/14)) |
    (reg["chair_completed"] == 0)
)
f0 = ((reg[["womac_R","womac_L"]].max(axis=1) <= 21.08) &
      (reg["walk_speed"] >= 1.26) &
      (reg["chair_completed"] == 1) & (reg["chair_pace"] >= 1/14))
reg["registry_class"] = np.where(~registry_required, np.nan,
                         np.where(severe, 2, np.where(f0, 0, 1)))
print(reg["registry_class"].value_counts(dropna=False).sort_index())

## 4. Build bilateral participant sequences and V03 → V06 landmark records

In [ ]:
VISIT_ORDER = ["V00","V01","V03","V05","V06","V08","V10"]
PRIMARY_LANDMARK, PRIMARY_TARGET = "V03", "V06"
# Keep actual month gaps. Require >=3 principal visits with core measurements for cohort inclusion.
core = long[long.visit.isin(VISIT_ORDER)].copy()
core_count = (core.dropna(subset=["walk_speed","chair_completed"])
              .groupby("participant_id").visit.nunique())
eligible_ids = core_count[core_count >= 3].index
core = core[core.participant_id.isin(eligible_ids)].copy()

# Convert to one row per participant-visit with bilateral outcomes and person covariates.
bilat = core.pivot_table(index=["participant_id","visit","month"],
                         columns="knee", values=knee_cols, aggfunc="first")
bilat.columns = ["{}_{}".format(a,b) for a,b in bilat.columns]
bilat = bilat.reset_index()
seq = person.merge(bilat, on=["participant_id","visit","month"], how="left")
seq = seq[seq.participant_id.isin(eligible_ids)].copy()
seq["visit_rank"] = seq.visit.map({v:i for i,v in enumerate(VISIT_ORDER)})
seq = seq.sort_values(["participant_id","month"])

# Merge registry class for transition target and history.
seq = seq.merge(reg[["participant_id","visit","month","registry_class"]],
                on=["participant_id","visit","month"], how="left")
# Actual V03 landmark and V06 target, no imputation of target outcomes.
hist = seq[seq.visit_rank <= VISIT_ORDER.index(PRIMARY_LANDMARK)].copy()
landmark = hist[hist.visit == PRIMARY_LANDMARK][["participant_id","month"]].rename(columns={"month":"landmark_month"})
target = seq[seq.visit == PRIMARY_TARGET].copy()
target = target.rename(columns={c: f"{c}_target" for c in target.columns if c != "participant_id"})
pairs = landmark.merge(target, on="participant_id", how="inner")
pairs["horizon_months"] = pairs["month_target"] - pairs["landmark_month"]
pairs = pairs[pairs.horizon_months > 0].copy()
print("Eligible participants:", len(eligible_ids), "Primary landmark-target pairs:", len(pairs))

## 5. Missingness-aware feature construction

In [ ]:
# Construct history features from observations available by V03 only.
# Numeric values are forward-filled within participant; missingness flags and time-since-observed
# are retained. No future interpolation is performed.
numeric_features = [c for c in [
    "womac_function_R","womac_function_L","womac_pain_R","womac_pain_L",
    "womac_stiffness_R","womac_stiffness_L","kl_grade_R","kl_grade_L","jsw_R","jsw_L",
    "walk_speed","chair_completed","chair_pace","bmi","comorbidity","physical_activity","age"
] if c in hist.columns]
categorical_features = [c for c in ["sex","medication"] if c in hist.columns]

def participant_history(pid):
    h = hist[hist.participant_id == pid].sort_values("month").copy()
    if h.empty: return None
    # Cap history to the three most recent observed principal assessments through V03.
    h = h.tail(3)
    vals=[]
    for _, r in h.iterrows():
        row=[]
        for c in numeric_features:
            row.extend([float(r[c]) if pd.notna(r[c]) else 0.0, float(pd.notna(r[c]))])
        for c in categorical_features:
            row.append(str(r[c]) if pd.notna(r[c]) else "UNKNOWN")
        row.extend([float(r["month"]), float(r["month"]-h["month"].iloc[0])])
        vals.append(row)
    # Left pad to length 3, with all-zero rows.
    dim=len(vals[0]); arr=np.zeros((3,dim),dtype=np.float32)
    arr[-len(vals):]=np.asarray(vals,dtype=np.float32) if not categorical_features else 0
    # categorical fields are encoded below in a simpler frequency-safe manner
    if categorical_features:
        # Replace categorical string entries by deterministic missing/known indicators.
        arr=np.zeros((3, 2*len(numeric_features)+2*len(categorical_features)+2),dtype=np.float32)
        for ti,(_,r) in enumerate(h.iterrows(), start=3-len(h)):
            z=[]
            for c in numeric_features: z += [float(r[c]) if pd.notna(r[c]) else 0., float(pd.notna(r[c]))]
            for c in categorical_features: z += [float(pd.notna(r[c])), 0.]
            z += [float(r["month"]),float(r["month"]-h["month"].iloc[0])]
            arr[ti]=z
    return arr

ids = pairs.participant_id.astype(str).tolist()
X = np.stack([participant_history(pid) for pid in ids])
# Numeric scaling over fitting participants only is performed after the participant split.
print("Sequence tensor:", X.shape)

## 6. Define forecast targets and transition/event labels

In [ ]:
# Forecast targets at V06. Side-specific WOMAC plus participant-level performance outcomes.
def col_or_nan(frame, name):
    return frame[name].to_numpy(dtype=float) if name in frame else np.full(len(frame), np.nan)
Y = np.column_stack([
    col_or_nan(pairs,"womac_function_R_target"),
    col_or_nan(pairs,"womac_function_L_target"),
    col_or_nan(pairs,"walk_speed_target"),
    col_or_nan(pairs,"chair_completed_target"),
    col_or_nan(pairs,"chair_pace_target")
]).astype(np.float32)
# Registry target class from measured target outcomes; unclassified remains NaN.
y_state = col_or_nan(pairs,"registry_class_target")
# Event labels from dated event records if supplied. Competing risks are event occurrence by target.
def event_by_target(event_col):
    if event_col not in pairs: return np.zeros(len(pairs),dtype=np.float32)
    ev=pd.to_datetime(pairs[event_col],errors="coerce")
    t=pd.to_numeric(pairs["month_target"],errors="coerce")
    # Event dates should be supplied as dates; compare date to target month only when available.
    return ev.notna().astype(float).to_numpy()
y_replace = event_by_target("replacement_date_target")
y_death = event_by_target("death_date_target")
print("Targets (n, outcomes):",Y.shape, "Observed target state:",np.isfinite(y_state).sum())

## 7. Participant-level train / calibration / locked test split

In [ ]:
all_idx=np.arange(len(ids))
fit_idx, hold_idx=train_test_split(all_idx,test_size=0.40,random_state=SEED)
cal_idx,test_idx=train_test_split(hold_idx,test_size=0.50,random_state=SEED)
print("Fit/calibration/test:",len(fit_idx),len(cal_idx),len(test_idx))
# Standardise each sequence feature using fitting-partition observations only.
flat=X[fit_idx].reshape(-1,X.shape[-1])
mu=flat.mean(axis=0); sd=flat.std(axis=0); sd[sd<1e-6]=1.
Xn=((X-mu)/sd).astype(np.float32)
# Robust target scaling for continuous heads, learned only on fitting data.
target_mu=np.nanmean(Y[fit_idx],axis=0)
target_sd=np.nanstd(Y[fit_idx],axis=0); target_sd[target_sd<1e-6]=1.
Yn=((Y-target_mu)/target_sd).astype(np.float32)

## 8. TGBAM-MSE encoder + probabilistic multi-task forecasting + transition heads

In [ ]:
class UADMobilityNet(nn.Module):
    def __init__(self, input_dim, hidden=96, latent=16):
        super().__init__()
        self.knee_encoder=nn.Sequential(nn.Linear(input_dim,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.shared_gru=nn.GRU(hidden,hidden,batch_first=True)
        self.knee_gru=nn.GRU(hidden,hidden,batch_first=True)
        self.asym_head=nn.Sequential(nn.Linear(hidden*3,hidden),nn.GELU(),nn.Dropout(.15))
        self.forecast=nn.Linear(hidden,10) # means/logvars for 5 outputs
        self.completion=nn.Linear(hidden,1)
        self.state=nn.Linear(hidden,3)
        self.events=nn.Linear(hidden,3) # replacement right, replacement left, death
        self.recovery=nn.Linear(hidden,2) # non-surgical / total observed-care recovery
        self.latent_dim=latent
        self.latent_load=nn.Linear(hidden,latent*5)
    def forward(self,x):
        # x: [B,T,F], one participant-level stream containing bilateral channels.
        e=self.knee_encoder(x)
        _,h=self.shared_gru(e)
        z=h[-1]
        # Side-equivariant proxy: bilateral signed channels are explicitly present in X.
        # Shared embedding + signed asymmetry channels from the sequence tensor.
        # This implementation shares all weights across sides via paired target heads.
        z=self.asym_head(torch.cat([z,z.abs(),z],dim=-1))
        raw=self.forecast(z).view(-1,5,2)
        means=raw[:,:,0]
        logvars=raw[:,:,1].clamp(-5,3)
        completion_logit=self.completion(z).squeeze(-1)
        state_logits=self.state(z)
        event_logits=self.events(z)
        recovery_logits=self.recovery(z)
        load=self.latent_load(z).view(-1,5,self.latent_dim)
        return means,logvars,completion_logit,state_logits,event_logits,recovery_logits,load

class TensorDS(Dataset):
    def __init__(self, idx):
        self.idx=idx
    def __len__(self): return len(self.idx)
    def __getitem__(self,k):
        i=self.idx[k]
        return (torch.tensor(Xn[i]),torch.tensor(Yn[i]),torch.tensor(y_state[i] if np.isfinite(y_state[i]) else -1),
                torch.tensor([y_replace[i],y_replace[i],y_death[i]],dtype=torch.float32))

## 9. Train with masked heteroscedastic forecasting and auxiliary state/event objectives

In [ ]:
model=UADMobilityNet(X.shape[-1]).to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
def masked_gaussian_nll(mu,logvar,y):
    mask=torch.isfinite(y)
    if mask.sum()==0: return torch.tensor(0.,device=DEVICE)
    return (0.5*(logvar+(y-mu)**2*torch.exp(-logvar)))[mask].mean()

def run_epoch(loader,train=True):
    model.train(train); losses=[]
    for xb,yb,sb,eb in loader:
        xb=xb.to(DEVICE); yb=yb.to(DEVICE); sb=sb.to(DEVICE); eb=eb.to(DEVICE)
        if train: optimizer.zero_grad()
        out=model(xb); mu,lv,cl,sl,el,rl,load=out
        loss=masked_gaussian_nll(mu,lv,yb)
        # Chair completion Bernoulli head is trained against completion target (column 3).
        cmask=torch.isfinite(yb[:,3])
        if cmask.any():
            loss=loss+nn.functional.binary_cross_entropy_with_logits(cl[cmask],yb[cmask,3].clamp(0,1))
        smask=sb>=0
        if smask.any(): loss=loss+0.25*nn.functional.cross_entropy(sl[smask],sb[smask].long())
        loss=loss+0.15*nn.functional.binary_cross_entropy_with_logits(el,eb)
        if train:
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),2.0); optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else np.nan

train_loader=DataLoader(TensorDS(fit_idx),batch_size=32,shuffle=True)
cal_loader=DataLoader(TensorDS(cal_idx),batch_size=128,shuffle=False)
EPOCHS=40
history=[]
for epoch in range(EPOCHS):
    tr=run_epoch(train_loader,True); history.append(tr)
    if (epoch+1)%5==0: print(f"Epoch {epoch+1:02d}/{EPOCHS} loss={tr:.4f}")
plt.figure(figsize=(7,4)); plt.plot(history); plt.xlabel("Epoch"); plt.ylabel("Training objective"); plt.title("UAD-Mobility training"); plt.grid(True); plt.show()

## 10. Ensemble uncertainty: aleatoric + epistemic (bootstrap members)

In [ ]:
# Train a compact participant-bootstrap ensemble. Each member is a complete paired model.
N_MEMBERS=5
members=[]
for m in range(N_MEMBERS):
    boot=np.random.default_rng(SEED+m).choice(fit_idx,size=len(fit_idx),replace=True)
    net=UADMobilityNet(X.shape[-1]).to(DEVICE)
    opt=torch.optim.AdamW(net.parameters(),lr=1e-3,weight_decay=1e-4)
    # A few epochs per member; increase after checking runtime and convergence.
    net.train()
    for ep in range(20):
        order=np.random.permutation(boot)
        for st in range(0,len(order),32):
            ii=order[st:st+32]
            xb=torch.tensor(Xn[ii],device=DEVICE); yb=torch.tensor(Yn[ii],device=DEVICE)
            out=net(xb); mu,lv,cl,sl,el,rl,load=out
            loss=masked_gaussian_nll(mu,lv,yb)
            cm=torch.isfinite(yb[:,3])
            if cm.any(): loss=loss+nn.functional.binary_cross_entropy_with_logits(cl[cm],yb[cm,3].clamp(0,1))
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(),2.); opt.step()
    members.append(net.eval())
print("Ensemble members:",len(members))

## 11. Prediction helper and conformal calibration (participant-level)

In [ ]:
@torch.no_grad()
def ensemble_predict(indices):
    xx=torch.tensor(Xn[indices],device=DEVICE)
    means=[]; vars_=[]; states=[]; events=[]; rec=[]
    for net in members:
        mu,lv,cl,sl,el,rl,load=net(xx)
        means.append(mu.cpu().numpy()); vars_.append(torch.exp(lv).cpu().numpy())
        states.append(torch.softmax(sl,dim=-1).cpu().numpy())
        events.append(torch.sigmoid(el).cpu().numpy()); rec.append(torch.sigmoid(rl).cpu().numpy())
    means=np.stack(means); vars_=np.stack(vars_)
    pred=means.mean(axis=0)
    aleatoric=vars_.mean(axis=0)
    epistemic=means.var(axis=0,ddof=1) if len(members)>1 else np.zeros_like(aleatoric)
    return {"mean":pred,"aleatoric":aleatoric,"epistemic":epistemic,
            "total_var":aleatoric+epistemic,"state":np.mean(states,axis=0),
            "events":np.mean(events,axis=0),"recovery":np.mean(rec,axis=0)}

cal_pred=ensemble_predict(cal_idx); test_pred=ensemble_predict(test_idx)
# 90% participant-level split conformal correction: max absolute standardised residual
# across both knees for primary bilateral WOMAC (columns 0 and 1).
alpha=0.10
cal_y=Yn[cal_idx]
cal_scale=np.sqrt(np.maximum(cal_pred["total_var"],1e-6))
cal_scores=np.nanmax(np.abs((cal_y[:,:2]-cal_pred["mean"][:,:2])/cal_scale[:,:2]),axis=1)
cal_scores=cal_scores[np.isfinite(cal_scores)]
q_idx=min(len(cal_scores)-1,int(np.ceil((len(cal_scores)+1)*(1-alpha)))-1)
qhat=float(np.sort(cal_scores)[q_idx])
print("90% bilateral participant conformal correction:",qhat)
test_scale=np.sqrt(np.maximum(test_pred["total_var"],1e-6))
lo=(test_pred["mean"]-qhat*test_scale)*target_sd+target_mu
hi=(test_pred["mean"]+qhat*test_scale)*target_sd+target_mu
pred_raw=test_pred["mean"]*target_sd+target_mu

## 12. Locked test evaluation: primary bilateral WOMAC and secondary outcomes

In [ ]:
yt=Y[test_idx]
def regression_metrics(y,p,mask=None):
    if mask is None: mask=np.isfinite(y)&np.isfinite(p)
    if mask.sum()==0: return {"n":0,"MAE":np.nan,"RMSE":np.nan,"R2":np.nan}
    return {"n":int(mask.sum()),"MAE":float(mean_absolute_error(y[mask],p[mask])),
            "RMSE":float(np.sqrt(mean_squared_error(y[mask],p[mask]))),
            "R2":float(r2_score(y[mask],p[mask])) if mask.sum()>1 else np.nan}
names=["Right WOMAC function","Left WOMAC function","20-m walk speed","Chair completion","Chair-stand pace"]
metrics={}
for j,n in enumerate(names):
    metrics[n]=regression_metrics(yt[:,j],pred_raw[:,j])
print(json.dumps(metrics,indent=2))
# Bilateral WOMAC interval coverage and average width.
test_lo=lo[:,:2]; test_hi=hi[:,:2]
mask=np.isfinite(yt[:,:2])
coverage=float(((yt[:,:2]>=test_lo)&(yt[:,:2]<=test_hi))[mask].mean()) if mask.any() else np.nan
width=float((test_hi-test_lo)[mask].mean()) if mask.any() else np.nan
print({"bilateral_WOMAC_marginal_coverage":coverage,"mean_interval_width":width,
       "nominal_coverage":1-alpha})

## 13. State and event-risk evaluation

In [ ]:
# State classification metrics where the target registry label is observed.
state_true=y_state[test_idx]
state_mask=np.isfinite(state_true)
if state_mask.any():
    state_hat=test_pred["state"][state_mask].argmax(axis=1)
    print(classification_report(state_true[state_mask].astype(int),state_hat,
                                labels=[0,1,2],zero_division=0))
    print("Confusion matrix:\n",confusion_matrix(state_true[state_mask].astype(int),state_hat,labels=[0,1,2]))
# Event outputs are predictive scores. If exact event-time/status variables are available,
# replace the provisional labels above with rigorously derived at-risk intervals and censoring.
event_prob=test_pred["events"]
print("Mean predicted replacement-right / replacement-left / death scores:",
      event_prob.mean(axis=0))

## 14. Reliability profile and calibration diagnostics

In [ ]:
total_sd=np.sqrt(np.maximum(test_pred["total_var"],1e-8))
profile=pd.DataFrame({
    "participant_id":np.asarray(ids,dtype=str)[test_idx],
    "WOMAC_R_total_SD":total_sd[:,0],
    "WOMAC_L_total_SD":total_sd[:,1],
    "WOMAC_R_aleatoric_SD":np.sqrt(test_pred["aleatoric"][:,0]),
    "WOMAC_R_epistemic_SD":np.sqrt(test_pred["epistemic"][:,0]),
    "predicted_F0":test_pred["state"][:,0],
    "predicted_F1":test_pred["state"][:,1],
    "predicted_F2":test_pred["state"][:,2],
    "replacement_R_score":event_prob[:,0],
    "replacement_L_score":event_prob[:,1],
    "death_score":event_prob[:,2],
})
display(profile.head(10))
plt.figure(figsize=(6,5))
plt.scatter(total_sd[:,0],np.abs(yt[:,0]-pred_raw[:,0]),alpha=.65)
plt.xlabel("Predicted total SD (right WOMAC)"); plt.ylabel("Absolute error")
plt.title("Uncertainty vs absolute error"); plt.grid(True); plt.show()

## 15. Side-swap equivariance sanity check

In [ ]:
# Swap explicitly paired feature columns in the raw source and verify that the model
# architecture's paired output interpretation is invariant to re-labelling.
# This diagnostic requires source-level R/L feature columns and is reported as a limitation
# if the canonical input tensor does not retain a clean bilateral feature pairing.
paired=[]
for c in numeric_features:
    if c.endswith("_R") and c[:-2]+"_L" in numeric_features:
        paired.append((c,c[:-2]+"_L"))
print("Detected bilateral feature pairs:",paired)
print("Note: for strict mathematical equivariance, retain side-keyed knee streams and use a shared knee cell;")
print("the compact baseline here uses shared sequence weights and paired outputs, and should be upgraded")
print("to the full side-keyed recurrent cell before claiming exact equivariance.")

## 16. Sensitivity analyses and ablations

In [ ]:
# A. Registry sensitivity: conventional slow-gait flag (<0.8 m/s) as a descriptive alternative.
if "walk_speed" in reg:
    reg["slow_gait_sensitivity"]=(reg.walk_speed<0.8).astype(float)
# B. Ensemble size sensitivity.
for k in [1, min(3,len(members)),len(members)]:
    pm=np.mean([ensemble_predict(test_idx)["mean"]],axis=0) if k==1 else None
    print(f"Ensemble-size configuration requested: {k}; retrain/aggregate first {k} members for formal comparison.")
# C. Report class prevalence and missingness.
print("Registry prevalence:")
print(reg.registry_class.value_counts(normalize=True,dropna=False).sort_index())
print("Target outcome missingness:")
print(pd.DataFrame(Y,columns=names).isna().mean())

## 17. Export predictions, reliability profiles, metrics and model weights

In [ ]:
out=pd.DataFrame({
    "participant_id":np.asarray(ids,dtype=str)[test_idx],
    "true_womac_R":yt[:,0],"pred_womac_R":pred_raw[:,0],
    "true_womac_L":yt[:,1],"pred_womac_L":pred_raw[:,1],
    "lower_womac_R":lo[:,0],"upper_womac_R":hi[:,0],
    "lower_womac_L":lo[:,1],"upper_womac_L":hi[:,1],
    "pred_walk_speed":pred_raw[:,2],
    "pred_chair_completion":1/(1+np.exp(-np.clip(pred_raw[:,3],-30,30))),
    "pred_chair_pace":pred_raw[:,4],
    "pred_F0":test_pred["state"][:,0],"pred_F1":test_pred["state"][:,1],"pred_F2":test_pred["state"][:,2],
    "risk_replacement_R":event_prob[:,0],"risk_replacement_L":event_prob[:,1],"risk_death":event_prob[:,2]
})
out.to_csv(os.path.join(OUTPUT_DIR,"locked_test_predictions.csv"),index=False)
profile.to_csv(os.path.join(OUTPUT_DIR,"reliability_profile.csv"),index=False)
with open(os.path.join(OUTPUT_DIR,"metrics.json"),"w") as f:
    json.dump({"regression":metrics,"bilateral_WOMAC_coverage":coverage,
               "mean_interval_width":width,"conformal_qhat":qhat},f,indent=2)
for i,net in enumerate(members):
    torch.save(net.state_dict(),os.path.join(OUTPUT_DIR,f"uad_mobility_member_{i+1}.pt"))
print("Saved outputs to:",OUTPUT_DIR)

## 18. Research validity checklist before reporting

1. Confirm the source OAI progression-subcohort extract and all visit codes, actual assessment dates, and variable versions.
2. Verify WOMAC scale direction/range and chair-stand pace definition against the source instrument.
3. Confirm whether replacement/death dates are available and distinguish exact event dates from visit-reported status.
4. Derive proper interval-specific risk sets and censoring for replacement/death. The compact event heads in this notebook are **risk-score baselines**, not the full monthly interval-censored semi-Markov likelihood described in Section 3.5.
5. Implement the exact zero-hurdle beta-binomial and latent Gaussian copula margins before claiming the exact Section 3.4 distribution family.
6. Implement the full side-keyed TGBAM-MSE recurrent cell and verify numerical side-swap equivariance before claiming exact Eq. (5)–(13).
7. Use participant-level cross-fitting, attendance/inverse-observation weighting, weighted conformal quantiles, and locked outer testing for the final paper. The notebook's split conformal baseline is unweighted and its coverage is marginal, not patient-conditional.
8. The supplied methods mention 1,390 participants; do not force this count. Report the number remaining after applying the prespecified eligibility and data-availability rules.
9. Treatment coefficients are predictive associations, not causal effects. This notebook does not estimate treatment effects.
10. Obtain appropriate ethical approval and governance for any clinician/patient-facing decision-support deployment.

**Do not interpret model outputs as diagnosis, prognosis for an individual, or treatment advice.**